In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [8]:
mattra=r"""

#include <cstdio>
#include <cuda_runtime.h>

# define TILE 32

__global__ void transpose(const float* in, float* out, int rows, int cols){
    int col = blockIdx.x*blockDim.x+threadIdx.x;
    int row = blockIdx.y*blockDim.y+threadIdx.y;

    if (row<rows && col<cols){
        out[col*rows+row]=in[row*cols+col];
    }
}

int main(){

    const int rows=100, cols=200;
    size_t bytes=rows*cols*sizeof(float);

    float *in, *out;
    cudaMallocManaged(&in, bytes);
    cudaMallocManaged(&out, bytes);

    for(int r=0; r<rows; r++){
        for(int c=0; c<cols; c++){
            in[r*cols+c]=r*1000+c;
        }
    }

    dim3 block(TILE, TILE);
    dim3 grid((cols+TILE-1)/TILE, (rows+TILE-1)/TILE);
    transpose<<<grid, block>>>(in, out, rows, cols);

    printf("in[3][5] = %.0f, out[5][3] = %.0f\n", in[3 * cols + 5], out[5 * rows + 3]);

    cudaFree(in);
    cudaFree(out);
    return 0;
}

"""

with open('transpose.cu', 'w') as f:
    f.write(mattra)

In [9]:
!ls

transpose.cu


In [10]:
!nvcc -arch=sm_75 transpose.cu -o mattra

In [11]:
!./mattra

in[3][5] = 3005, out[5][3] = 0
